In [10]:
import pandas as pd
import os
import ast
# Rutas de los archivos
Reporte_Imprevistas_05 = "C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\GEMAS-RETIN\IMPUT-TOTAL\IMPREVISTAS-TOTAL.xlsx"

# Importar archivos
# TXT (ajusta el separador si es necesario)

df_Reporte_Imprevistas_05= pd.read_excel(Reporte_Imprevistas_05,sheet_name="Hoja1")
df_Reporte_Imprevistas_05["GEMAS"] = df_Reporte_Imprevistas_05["GEMAS"].astype(str)
df_Reporte_Imprevistas_05 =df_Reporte_Imprevistas_05.rename(columns={"GEMAS": "referencia"})
display(df_Reporte_Imprevistas_05)

,DIA,referencia,REF CC,NT,UOT,Zona,CFT,Subestación,LMT,Estado,Fecha inicio-hora,Fecha Fin,Tipo Evento,Causa,Ticket,Cl. Afec.,G.res
0,1,B077663,B077663,NaN,MT,Norte Chico,SANTA CRUZ DE ANDAMARCA,Ravira Pacaraos,RP-01,CERRADO,2026-03-01 00:02:00,2026-03-01 00:04:00,DESCONEXION MANUAL O MANIOBRAS,APERTURA DE RP-01\nRETIRO DE G. ELECTROGENO E ...,494,DF202681717359,P
1,1,B077667,B077667,NaN,MT,Panamericana,SANTA ROSA,Ancon,N-06,CERRADO,2026-03-01 01:28:51,2026-03-01 17:12:00,AVERIA,INTERRUPCIÓN IMPREVISTA PARCIAL EN N-06\nAPERT...,1491,DF202681717364,T
2,1,B077668,B077668,NaN,MT,Norte Chico,PACHANGARA,Nava,NV-01,CERRADO,2026-03-01 01:30:00,2026-03-01 01:30:05,DESCONEXION MANUAL O MANIOBRAS,APERTURA DE NV-01 POR TRABAJO DE OPERACIÓN.\nT...,963,DF202681717366,P
3,1,PM-TMP-260301100516,PM-TMP-260301100516,NaN,MT,Norte Chico,HUALMAY,Huacho,H-03,CERRADO,2026-03-01 07:00:00,2026-03-01 10:22:00,AVERIA,INTERRUPCIÓN IMPREVISTA PUNTUAL EN H-03\nAPERT...,138,DF202681717558,P
4,1,B077686,B077686,NaN,MT,Norte Chico,PACHANGARA,Nava,NV-01,CERRADO,2026-03-01 07:00:00,2026-03-01 07:00:05,AVERIA,APERTURA DE NV-01 POR SOBRECARGA.\nTRANSFERENC...,891,SIN TICKET GESI,P
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
523,29,B079380,B079380,NaN,MT,COLONIAL,SAN JUAN DE LURIGANCHO,CANTO GRANDE,CG-12,CERRADO,2026-03-29 13:01:00,2026-03-29 14:35:55,AVERIA,INTERRUPCIÓN IMPREVISTA PARCIAL EN CG-12\nAPER...,DF202681745406,P,757.75
524,29,nan,PM-TMP-260329132733,NaN,MT,PANAMERICANA,PUENTE PIEDRA,PUENTE PIEDRA,PP-02,CERRADO,2026-03-29 13:25:52,2026-03-29 14:48:16,AVERIA,INTERRUPCIÓN IMPREVISTA PARCIAL EN PP-02\nAPER...,DF202681745417,P,2072.05
525,29,B079393,B079393,NaN,MT,PANAMERICANA,LA PERLA,SANTA MARINA,F-04,CERRADO,2026-03-29 17:19:00,2026-03-30 05:54:00,AVERIA,INTERRUPCIÓN IMPREVISTA PUNTUAL EN F-04\nAPERT...,DF202681745754,P,704.648
526,29,B079390,B079390,NaN,MT,COLONIAL,LURIGANCHO-CHOSICA,JICAMARCA,J-09,CERRADO,2026-03-29 18:18:30,2026-03-29 19:44:00,AVERIA,INTERRUPCIÓN IMPREVISTA TOTAL EN J-09\nAPERTUR...,DF202681745559,T,12491.95


In [11]:
import pandas as pd
import os
import ast

# -----------------------------
# 1. Cargar los archivos
# -----------------------------
file_gemas = r"C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\GEMAS-RETIN\IMPUT-TOTAL\Report_Desenergización.xlsx"
file_informe = r"C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\GEMAS-RETIN\IMPUT-TOTAL\Informe_evento.txt"
file_vclientes = r"C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\GEMAS-RETIN\IMPUT-TOTAL\V_CLIENTES.txt"

# GEMAS
df_gemas = pd.read_excel(file_gemas, sheet_name='Report_Desenergización')
df_gemas.rename(columns={
    'Referencia': 'referencia',
    'Fecha Inicio Etapa': 'FECHA_INICIO',
    'Fecha Fin Etapa': 'FECHA_FIN',
    'CD': 'COD_SED'
}, inplace=True)

# INFORME_EVENTO
df_informe = pd.read_csv(file_informe, sep="$")
df_informe = df_informe[df_informe['Estado'].isin(['Validado', 'Cumplimentado', 'Cerrado','Activo'])]
df_gemas = df_gemas[df_gemas['referencia'].isin(df_informe['Referencia'])]

# Agrupar GEMAS
df_gemas_grouped = df_gemas.groupby(['referencia', 'FECHA_INICIO', 'FECHA_FIN'])['COD_SED']\
    .apply(list).reset_index().rename(columns={'COD_SED': 'SEDS_AFECTADAS'})

# Verificar el formato de las fechas
print("Ejemplo de FECHA_INICIO:", df_gemas_grouped['FECHA_INICIO'].iloc[0])
print("Ejemplo de FECHA_FIN:", df_gemas_grouped['FECHA_FIN'].iloc[0])

# Luego aplicar la conversión correcta
if "-" in str(df_gemas_grouped['FECHA_INICIO'].iloc[0]):
    # Formato con guiones: "dd-mm-yyyy HH:MM:SS.sss"
    df_gemas_grouped['FECHA_INICIO'] = pd.to_datetime(df_gemas_grouped['FECHA_INICIO'], format="%d-%m-%Y %H:%M:%S.%f", dayfirst=True).dt.floor('s')
    df_gemas_grouped['FECHA_FIN'] = pd.to_datetime(df_gemas_grouped['FECHA_FIN'], format="%d-%m-%Y %H:%M:%S.%f", dayfirst=True).dt.floor('s')
else:
    # Formato con barras: "dd/mm/yyyy HH:MM:SS.sss"
    df_gemas_grouped['FECHA_INICIO'] = pd.to_datetime(df_gemas_grouped['FECHA_INICIO'], format="%d/%m/%Y %H:%M:%S.%f", dayfirst=True).dt.floor('s')
    df_gemas_grouped['FECHA_FIN'] = pd.to_datetime(df_gemas_grouped['FECHA_FIN'], format="%d/%m/%Y %H:%M:%S.%f", dayfirst=True).dt.floor('s')

df_gemas_grouped['DURACION_MIN'] = (df_gemas_grouped['FECHA_FIN'] - df_gemas_grouped['FECHA_INICIO']).dt.total_seconds() / 60



# Separar comparables y no comparables
df_comparables = df_gemas_grouped[df_gemas_grouped['DURACION_MIN'] > 3].copy()
df_no_comparados = df_gemas_grouped[df_gemas_grouped['DURACION_MIN'] <= 3].copy()

# -----------------------------
# 2. Procesar HENRY (etapas)

#------------------------------------------------------------
# Definir los anchos de columna en el orden exacto que aparecen
col_widths = [65, 20, 20, 26, 101, 101, 101, 101, 101, 105, 7, 2]

# Definir los nombres de columna en el mismo orden
col_names = [
    'IDINTA',
    'DATA_DISALIMENTAZIO',  # Note: different from your rename key
    'DATA_ALIMENTAZIONE1',
    'NUM_CLIENTI_DISALIMENTATI',
    'NOME_ZONA',
    'NOME_CS',
    'NOME_SBCS_ENEL',
    'NOME_CP',
    'NOME_LINEA_MT',
    'NOME_SBCS_RETE',
    'NUMUTE',
    'NUMUTE '  # Note: space at the end
]

# Leer el archivo de ancho fijo
df_vclientes = pd.read_fwf(
    file_vclientes,
    widths=col_widths,
    names=col_names,  # Si el archivo NO tiene encabezado
    header=None,      # Especificar que no hay encabezado
    encoding='utf-8'  # Ajustar encoding según sea necesario
)

# Si el archivo SÍ tiene encabezado (primera fila con nombres)
df_vclientes = pd.read_fwf(file_vclientes, widths=col_widths, encoding='utf-8')
df_vclientes = df_vclientes.iloc[1:].reset_index(drop=True)

# Renombrar columnas según lo que necesitas
df_vclientes.rename(columns={
    'IDINTA': 'idinta',
    'DATA_DISALIMENTAZIO': 'FECHA_INICIO',  # Note: usando el nombre real
    'DATA_ALIMENTAZIONE1': 'FECHA_FIN',
    'NOME_SBCS_ENEL': 'COD_SED'
}, inplace=True)
#------------------------------------------------------------

display(df_vclientes)

df_vclientes['FECHA_INICIO'] = pd.to_datetime(df_vclientes['FECHA_INICIO'], format='%d/%m/%Y %H:%M:%S', errors='coerce')
df_vclientes['FECHA_FIN'] = pd.to_datetime(df_vclientes['FECHA_FIN'], format='%d/%m/%Y %H:%M:%S', errors='coerce')

df_vclientes['FECHA_INICIO'] = pd.to_datetime(df_vclientes['FECHA_INICIO']).dt.floor('s')
df_vclientes['FECHA_FIN'] = pd.to_datetime(df_vclientes['FECHA_FIN']).dt.floor('s')


etapas = []
for idinta, grupo in df_vclientes.groupby('idinta'):
    grupo = grupo.sort_values(by=['FECHA_INICIO', 'FECHA_FIN']).reset_index(drop=True)
    activos = grupo.copy()
    tiempo_actual = grupo['FECHA_INICIO'].min()
    while not activos.empty:
        sed_en_falla = activos[activos['FECHA_FIN'] > tiempo_actual]
        if sed_en_falla.empty:
            break
        seds_afectadas = sed_en_falla['COD_SED'].tolist()
        tiempo_fin = sed_en_falla['FECHA_FIN'].min()
        etapas.append({
            'idinta': idinta,
            'FECHA_INICIO': tiempo_actual,
            'FECHA_FIN': tiempo_fin,
            'SEDS_AFECTADAS': seds_afectadas
        })
        activos = activos[activos['FECHA_FIN'] > tiempo_fin]
        tiempo_actual = tiempo_fin

df_henry_etapas = pd.DataFrame(etapas)
display(df_henry_etapas)
# -----------------------------
# 3. Comparación
# -----------------------------
def comparar_evento(row, henry_df):
    fecha_inicio = row['FECHA_INICIO']
    fecha_fin = row['FECHA_FIN']
    seds_gemas = set(row['SEDS_AFECTADAS'])
    coincidencias = henry_df[
        (henry_df['FECHA_INICIO'] == fecha_inicio) &
        (henry_df['FECHA_FIN'] == fecha_fin) &
        (henry_df['SEDS_AFECTADAS'].apply(lambda x: set(x) == seds_gemas))
    ]
    return "CORRECTO" if not coincidencias.empty else "ERRÓNEO"

def detalle_error(row, henry_df):
    if row['Estado'] == 'CORRECTO':
        return ""
    fecha_inicio = row['FECHA_INICIO']
    fecha_fin = row['FECHA_FIN']
    seds_gemas = set(row['SEDS_AFECTADAS'])
    coincidencia_horario = henry_df[
        (henry_df['FECHA_INICIO'] == fecha_inicio) &
        (henry_df['FECHA_FIN'] == fecha_fin)
    ]
    if coincidencia_horario.empty:
        return "Diferencia de horario"
    elif any(set(match['SEDS_AFECTADAS']) != seds_gemas for _, match in coincidencia_horario.iterrows()):
        return "Diferencia en SEDs"
    else:
        return "Diferencia de horario y SEDs"

df_comparacion = df_comparables.copy()
df_comparacion['Estado'] = df_comparacion.apply(lambda row: comparar_evento(row, df_henry_etapas), axis=1)
df_comparacion['Detalle Error'] = df_comparacion.apply(lambda row: detalle_error(row, df_henry_etapas), axis=1)
# -----------------------------
display(df_comparacion)


# Para poder hacer el merge por 'SEDS_AFECTADAS', primero convierte las listas a tuplas (que son hashables)
df_henry_etapas['SEDS_AFECTADAS'] = df_henry_etapas['SEDS_AFECTADAS'].apply(lambda x: tuple(sorted(x)))
#df_comparacion['SEDS_AFECTADAS'] = df_comparacion['SEDS_AFECTADAS'].apply(lambda x: tuple(sorted(x)))
df_comparacion['SEDS_AFECTADAS'] = df_comparacion['SEDS_AFECTADAS'].apply(
    lambda x: tuple(sorted(map(str, x))) if isinstance(x, list) else tuple()
)




df_comparacion_acu = pd.merge(df_henry_etapas, df_comparacion, on=["SEDS_AFECTADAS", "FECHA_INICIO", "FECHA_FIN"], how="outer", indicator=True)

# Para los eventos no comparados, también convierte la columna para mantener consistencia
df_no_comparados['SEDS_AFECTADAS'] = df_no_comparados['SEDS_AFECTADAS'].apply(lambda x: tuple(sorted(x)))
df_no_comparacion_acu = df_no_comparados.copy()
print("df_comparacion_acu")
display(df_comparacion_acu)

# 4. Exportar archivos
# -----------------------------
output_dir = os.path.dirname(r"C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\GEMAS-RETIN\OUTPUT-TOTAL")
df_comparacion_acu.to_excel(os.path.join(output_dir, "COMPARACION FINAL GEMAS VS HENRY.xlsx"), index=False)
df_no_comparacion_acu.to_excel(os.path.join(output_dir, "EVENTOS NO COMPARADOS - DURACION MENORES A 3 MIN.xlsx"), index=False)

df_comparacion_acu[df_comparacion_acu['Estado'] == 'ERRÓNEO'].to_excel(
    os.path.join(output_dir, "EVENTOS ERRONEOS - NO COINCIDEN CON HENRY.xlsx"), index=False)

print("✅ Proceso completo y archivos generados.")






Ejemplo de FECHA_INICIO: 2026-03-01 00:02:00
Ejemplo de FECHA_FIN: 2026-03-01 00:04:00


,idinta,FECHA_INICIO,FECHA_FIN,NUM_CLIENTI_DISALIMENTATI,NOME_ZONA,NOME_CS,COD_SED,NOME_CP,NOME_LINEA_MT,NOME_SBCS_RETE,NUMUTE,TI
0,EE1026000061,02/01/2026 14:11:41,02/01/2026 14:13:21,304,Colonial,04865A,04865A,CTO_GRANDE,CG-14,04865A,304,B
1,EE1026000061,02/01/2026 14:13:59,02/01/2026 17:07:36,304,Colonial,04865A,04865A,CTO_GRANDE,CG-14,04865A,304,B
2,EE1026000061,02/01/2026 14:11:41,02/01/2026 14:13:21,71,Colonial,06412C,06412C,CTO_GRANDE,CG-14,06412C,71,B
3,EE1026000061,02/01/2026 16:22:55,02/01/2026 17:07:36,71,Colonial,06412C,06412C,CTO_GRANDE,CG-14,06412C,71,B
4,EE1026000061,02/01/2026 14:11:41,02/01/2026 14:13:21,39,Colonial,05397C,05397C,CTO_GRANDE,CG-14,05397C,39,B
...,...,...,...,...,...,...,...,...,...,...,...,...
43935,EE1026013623,29/03/2026 09:44:42,29/03/2026 14:51:24,1,Panamericana,PM0433,PM0433,T-VALLE,TV-10,PM0433,1,M
43936,EE1026013633,29/03/2026 07:02:00,29/03/2026 20:00:00,82,Colonial,04029A,04029A,IZAGUIRRE,IE-08,04029A,82,B
43937,EE1026013651,29/03/2026 13:01:00,29/03/2026 14:35:55,111,Colonial,03959A,03959A,CTO_GRANDE,CG-12,03959A,111,B
43938,EE1026013651,29/03/2026 13:01:00,29/03/2026 14:35:55,178,Colonial,04077A,04077A,CTO_GRANDE,CG-12,04077A,178,B


,idinta,FECHA_INICIO,FECHA_FIN,SEDS_AFECTADAS
0,EE0026000127,2026-01-03 21:49:29,2026-01-03 21:51:47,"[22580U, 00503S, 15881A, 13141A, 06208C, PM211..."
1,EE0026000217,2026-01-05 08:33:57,2026-01-05 08:34:00,"[52051U, 52046U, 52048U, 52049U, 52058U, 51549..."
2,EE0026000217,2026-01-05 08:34:00,2026-01-05 08:34:25,"[51549U, 51758U, 51543U, 51804U, 51837U, 51518..."
3,EE0026000259,2026-01-05 13:09:14,2026-01-05 13:09:36,"[51786U, 51048U, 01082S, 51046U, 00339S, 02416..."
4,EE0026000567,2026-01-08 15:26:15,2026-01-08 15:41:50,"[36871U, 01077S, 07443C, 07442C, 01121S]"
...,...,...,...,...
4115,EE1026013747,2026-03-30 14:49:14,2026-03-30 14:50:03,"[02199A, 03253A, 04704A, 03405A, 04705A, 03415..."
4116,EE1026013749,2026-03-30 10:05:00,2026-03-30 15:00:00,[16561A]
4117,EE1026013755,2026-03-30 10:05:00,2026-03-30 10:06:00,[16151A]
4118,EE1026013757,2026-03-30 15:03:38,2026-03-30 15:07:18,"[51145U, PM0006, 51149U, 51727U, 00772S]"


,referencia,FECHA_INICIO,FECHA_FIN,SEDS_AFECTADAS,DURACION_MIN,Estado,Detalle Error
1,B077667,2026-03-01 01:28:51,2026-03-01 02:57:07,"[16297A, 16020A, 04733A, PM1488, 04895A, 02874...",88.266667,CORRECTO,
2,B077667,2026-03-01 02:57:07,2026-03-01 03:21:35,"[16297A, 16020A, 04733A, PM1488, 04895A, 02874...",24.466667,CORRECTO,
3,B077667,2026-03-01 03:21:35,2026-03-01 05:53:51,"[16297A, 16020A, 04733A, PM1488, 04895A, 02874...",152.266667,CORRECTO,
4,B077667,2026-03-01 05:53:51,2026-03-01 17:12:00,[16297A],678.150000,CORRECTO,
5,B077673,2026-03-01 04:23:40,2026-03-01 14:29:26,"[14364A, 01698S, 03347A, 03350A, 16127A, 06210...",605.766667,ERRÓNEO,Diferencia de horario
...,...,...,...,...,...,...,...
1343,B079391,2026-03-29 22:07:03,2026-03-29 22:21:00,"[14388A, 14968A, 14196A, 14774A, 14941A, 13639...",13.950000,ERRÓNEO,Diferencia de horario
1344,B079391,2026-03-29 22:21:00,2026-03-29 22:28:09,"[14196A, 14774A, 13639A, 13149A, 14756A, PM201...",7.150000,ERRÓNEO,Diferencia de horario
1345,B079393,2026-03-29 17:19:00,2026-03-30 00:55:22,[03041A],456.366667,ERRÓNEO,Diferencia de horario
1346,B079393,2026-03-30 00:55:22,2026-03-30 01:00:52,"[03040A, 03041A, 06138C, 06139C, 03045A, 03038...",5.500000,ERRÓNEO,Diferencia en SEDs


df_comparacion_acu


,idinta,FECHA_INICIO,FECHA_FIN,SEDS_AFECTADAS,referencia,DURACION_MIN,Estado,Detalle Error,_merge
0,EE0026002463,2026-01-22 02:42:27,2026-01-22 02:42:41,"(00001S, 00004S, 00007S, 00007S, 00007S, 00011...",NaN,NaN,NaN,NaN,left_only
1,NaN,2026-03-03 10:21:53,2026-03-03 17:19:22,"(00002S, 00003S, 00004S, 00005S, 00010S, 00010...",B077845-01,417.483333,ERRÓNEO,Diferencia de horario,right_only
2,EE0026002443,2026-01-21 23:46:21,2026-01-21 23:46:40,"(00002S, 00005S, 00010S, 00010S, 00012S, 00012...",NaN,NaN,NaN,NaN,left_only
3,EE0026003681,2026-01-30 00:01:00,2026-01-30 00:01:49,"(00002S, 00005S, 00010S, 00010S, 00012S, 00012...",NaN,NaN,NaN,NaN,left_only
4,EE0026003681,2026-01-30 00:01:49,2026-01-30 00:01:50,"(00002S, 00005S, 00010S, 00010S, 00012S, 00012...",NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...
4537,EE1026005021,2026-02-07 08:04:00,2026-02-07 11:33:14,"(PM2127,)",NaN,NaN,NaN,NaN,left_only
4538,EE1026006283,2026-02-15 14:50:00,2026-02-15 15:08:00,"(SEAUX2,)",NaN,NaN,NaN,NaN,left_only
4539,EE1026005461,2026-02-10 11:22:03,2026-02-10 21:14:17,"(SEAUX2, ZIG-ZAG)",NaN,NaN,NaN,NaN,left_only
4540,EE1026006155,2026-02-15 02:06:30,2026-02-15 02:09:00,"(SEAUX2, ZIG-ZAG)",NaN,NaN,NaN,NaN,left_only


✅ Proceso completo y archivos generados.


In [12]:
df_comparacion_Total =pd.merge(df_comparacion_acu,df_Reporte_Imprevistas_05, on="referencia", how="outer")
display(df_comparacion_Total)


,idinta,FECHA_INICIO,FECHA_FIN,SEDS_AFECTADAS,referencia,DURACION_MIN,Estado_x,Detalle Error,_merge,DIA,...,Subestación,LMT,Estado_y,Fecha inicio-hora,Fecha Fin,Tipo Evento,Causa,Ticket,Cl. Afec.,G.res
0,NaN,NaT,NaT,NaN,B020326J-09,NaN,NaN,NaN,NaN,2.0,...,Jicamarca,J-09,CERRADO,2026-03-02 05:25:00,2026-03-02 10:58:00,AVERIA,INTERRUPCIÓN IMPREVISTA PUNTUAL EN J-09\nAPERT...,66,DF202681718012,P
1,NaN,NaT,NaT,NaN,B02362A,NaN,NaN,NaN,NaN,5.0,...,Ventanilla,V-08,CERRADO,2026-03-05 12:43:00,2026-03-05 15:16:00,DESCONEXION MANUAL O MANIOBRAS,CORTE POR EMERGENCIA EN V-08\nAPERTURA DE 0236...,300,DF202681721733,P
2,NaN,NaT,NaT,NaN,B030326CV-03,NaN,NaN,NaN,NaN,3.0,...,Caudivilla,CV-03,CERRADO,2026-03-03 16:16:00,2026-03-03 18:07:00,DESCONEXION MANUAL O MANIOBRAS,CORTE POR EMERGENCIA EN CV-03\nAPERTURA DE 026...,2,DF202681719838,P
3,NaN,NaT,NaT,NaN,B077663,NaN,NaN,NaN,NaN,1.0,...,Ravira Pacaraos,RP-01,CERRADO,2026-03-01 00:02:00,2026-03-01 00:04:00,DESCONEXION MANUAL O MANIOBRAS,APERTURA DE RP-01\nRETIRO DE G. ELECTROGENO E ...,494,DF202681717359,P
4,NaN,2026-03-01 01:28:51,2026-03-01 02:57:07,"(02874A, 04733A, 04893A, 04894A, 04895A, 04896...",B077667,88.266667,CORRECTO,,right_only,1.0,...,Ancon,N-06,CERRADO,2026-03-01 01:28:51,2026-03-01 17:12:00,AVERIA,INTERRUPCIÓN IMPREVISTA PARCIAL EN N-06\nAPERT...,1491,DF202681717364,T
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4745,EE1026005021,2026-02-07 08:04:00,2026-02-07 11:33:14,"(PM2127,)",NaN,NaN,NaN,NaN,left_only,NaN,...,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN
4746,EE1026006283,2026-02-15 14:50:00,2026-02-15 15:08:00,"(SEAUX2,)",NaN,NaN,NaN,NaN,left_only,NaN,...,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN
4747,EE1026005461,2026-02-10 11:22:03,2026-02-10 21:14:17,"(SEAUX2, ZIG-ZAG)",NaN,NaN,NaN,NaN,left_only,NaN,...,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN
4748,EE1026006155,2026-02-15 02:06:30,2026-02-15 02:09:00,"(SEAUX2, ZIG-ZAG)",NaN,NaN,NaN,NaN,left_only,NaN,...,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:

import pandas as pd
from io import StringIO
df_comparacion_Total

# Función que decide el tipo de revisión por grupo
def clasificar_estado(grupo):
    estados = grupo["Estado_x"].unique()
    if all(est == "ERRÓNEO" for est in estados):
        return pd.Series(["REVISAR TOTALMENTE"] * len(grupo), index=grupo.index)
    elif all(est == "CORRECTO" for est in estados):
        return pd.Series(["VALIDAR"] * len(grupo), index=grupo.index)
    else:
        return pd.Series(["REVISAR PARCIALMENTE"] * len(grupo), index=grupo.index)



df_comparacion_Total["Revision"] = df_comparacion_Total.groupby("referencia").apply(clasificar_estado).reset_index(level=0, drop=True)

display(df_comparacion_Total)

 # Crear una nueva columna para inicializar el resultado
df_comparacion_Total["Revision_vs_idinta"] = "Faltan Intervalos"

 # Obtener todas las claves únicas
claves = df_comparacion_Total["referencia"].unique()

for clave in claves:
    # Filtrar por grupo 'referencia'
    grupo_ref = df_comparacion_Total[df_comparacion_Total["referencia"] == clave]
    # Obtener los valores únicos de Revision asociados a esa referencia
    revision_ref = grupo_ref["Revision"].unique()

    # Filtrar por grupo 'idinta' igual a la clave actual
    grupo_idinta = df_comparacion_Total[df_comparacion_Total["idinta"] == clave]
    revision_idinta = grupo_idinta["Revision"].unique()

    # Comparar valores únicos (orden no importa)
    if set(revision_ref) == set(revision_idinta):
        df_comparacion_Total.loc[df_comparacion_Total["idinta"] == clave, "Revision_vs_idinta"] = "VALIDAR"

# Aplicar la función por grupo y crear una nueva columna
df_comparacion_Total["Revision"] = df_comparacion_Total.groupby("referencia").apply(clasificar_estado).reset_index(level=0, drop=True)


# Mostrar resultado
display(df_comparacion_Total)

# Mostrar resultado
display(df_comparacion_Total)

df_comparacion_Total.to_excel(
    r"C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\GEMAS-RETIN\OUTPUT-TOTAL\COMPARACION FINAL GEMAS VS HENRY-CON IMPREVISTAS.xlsx",
    index=False,
    engine="openpyxl"  # Asegura que se guarde correctamente en formato .xlsx
)

C:\Users\P760033411\AppData\Local\Temp\ipykernel_1696\2412872040.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_comparacion_Total["Revision"] = df_comparacion_Total.groupby("referencia").apply(clasificar_estado).reset_index(level=0, drop=True)


,idinta,FECHA_INICIO,FECHA_FIN,SEDS_AFECTADAS,referencia,DURACION_MIN,Estado_x,Detalle Error,_merge,DIA,...,LMT,Estado_y,Fecha inicio-hora,Fecha Fin,Tipo Evento,Causa,Ticket,Cl. Afec.,G.res,Revision
0,NaN,NaT,NaT,NaN,B020326J-09,NaN,NaN,NaN,NaN,2.0,...,J-09,CERRADO,2026-03-02 05:25:00,2026-03-02 10:58:00,AVERIA,INTERRUPCIÓN IMPREVISTA PUNTUAL EN J-09\nAPERT...,66,DF202681718012,P,REVISAR PARCIALMENTE
1,NaN,NaT,NaT,NaN,B02362A,NaN,NaN,NaN,NaN,5.0,...,V-08,CERRADO,2026-03-05 12:43:00,2026-03-05 15:16:00,DESCONEXION MANUAL O MANIOBRAS,CORTE POR EMERGENCIA EN V-08\nAPERTURA DE 0236...,300,DF202681721733,P,REVISAR PARCIALMENTE
2,NaN,NaT,NaT,NaN,B030326CV-03,NaN,NaN,NaN,NaN,3.0,...,CV-03,CERRADO,2026-03-03 16:16:00,2026-03-03 18:07:00,DESCONEXION MANUAL O MANIOBRAS,CORTE POR EMERGENCIA EN CV-03\nAPERTURA DE 026...,2,DF202681719838,P,REVISAR PARCIALMENTE
3,NaN,NaT,NaT,NaN,B077663,NaN,NaN,NaN,NaN,1.0,...,RP-01,CERRADO,2026-03-01 00:02:00,2026-03-01 00:04:00,DESCONEXION MANUAL O MANIOBRAS,APERTURA DE RP-01\nRETIRO DE G. ELECTROGENO E ...,494,DF202681717359,P,REVISAR PARCIALMENTE
4,NaN,2026-03-01 01:28:51,2026-03-01 02:57:07,"(02874A, 04733A, 04893A, 04894A, 04895A, 04896...",B077667,88.266667,CORRECTO,,right_only,1.0,...,N-06,CERRADO,2026-03-01 01:28:51,2026-03-01 17:12:00,AVERIA,INTERRUPCIÓN IMPREVISTA PARCIAL EN N-06\nAPERT...,1491,DF202681717364,T,VALIDAR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4745,EE1026005021,2026-02-07 08:04:00,2026-02-07 11:33:14,"(PM2127,)",NaN,NaN,NaN,NaN,left_only,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4746,EE1026006283,2026-02-15 14:50:00,2026-02-15 15:08:00,"(SEAUX2,)",NaN,NaN,NaN,NaN,left_only,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4747,EE1026005461,2026-02-10 11:22:03,2026-02-10 21:14:17,"(SEAUX2, ZIG-ZAG)",NaN,NaN,NaN,NaN,left_only,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4748,EE1026006155,2026-02-15 02:06:30,2026-02-15 02:09:00,"(SEAUX2, ZIG-ZAG)",NaN,NaN,NaN,NaN,left_only,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN


C:\Users\P760033411\AppData\Local\Temp\ipykernel_1696\2412872040.py:42: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_comparacion_Total["Revision"] = df_comparacion_Total.groupby("referencia").apply(clasificar_estado).reset_index(level=0, drop=True)


,idinta,FECHA_INICIO,FECHA_FIN,SEDS_AFECTADAS,referencia,DURACION_MIN,Estado_x,Detalle Error,_merge,DIA,...,Estado_y,Fecha inicio-hora,Fecha Fin,Tipo Evento,Causa,Ticket,Cl. Afec.,G.res,Revision,Revision_vs_idinta
0,NaN,NaT,NaT,NaN,B020326J-09,NaN,NaN,NaN,NaN,2.0,...,CERRADO,2026-03-02 05:25:00,2026-03-02 10:58:00,AVERIA,INTERRUPCIÓN IMPREVISTA PUNTUAL EN J-09\nAPERT...,66,DF202681718012,P,REVISAR PARCIALMENTE,Faltan Intervalos
1,NaN,NaT,NaT,NaN,B02362A,NaN,NaN,NaN,NaN,5.0,...,CERRADO,2026-03-05 12:43:00,2026-03-05 15:16:00,DESCONEXION MANUAL O MANIOBRAS,CORTE POR EMERGENCIA EN V-08\nAPERTURA DE 0236...,300,DF202681721733,P,REVISAR PARCIALMENTE,Faltan Intervalos
2,NaN,NaT,NaT,NaN,B030326CV-03,NaN,NaN,NaN,NaN,3.0,...,CERRADO,2026-03-03 16:16:00,2026-03-03 18:07:00,DESCONEXION MANUAL O MANIOBRAS,CORTE POR EMERGENCIA EN CV-03\nAPERTURA DE 026...,2,DF202681719838,P,REVISAR PARCIALMENTE,Faltan Intervalos
3,NaN,NaT,NaT,NaN,B077663,NaN,NaN,NaN,NaN,1.0,...,CERRADO,2026-03-01 00:02:00,2026-03-01 00:04:00,DESCONEXION MANUAL O MANIOBRAS,APERTURA DE RP-01\nRETIRO DE G. ELECTROGENO E ...,494,DF202681717359,P,REVISAR PARCIALMENTE,Faltan Intervalos
4,NaN,2026-03-01 01:28:51,2026-03-01 02:57:07,"(02874A, 04733A, 04893A, 04894A, 04895A, 04896...",B077667,88.266667,CORRECTO,,right_only,1.0,...,CERRADO,2026-03-01 01:28:51,2026-03-01 17:12:00,AVERIA,INTERRUPCIÓN IMPREVISTA PARCIAL EN N-06\nAPERT...,1491,DF202681717364,T,VALIDAR,Faltan Intervalos
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4745,EE1026005021,2026-02-07 08:04:00,2026-02-07 11:33:14,"(PM2127,)",NaN,NaN,NaN,NaN,left_only,NaN,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Faltan Intervalos
4746,EE1026006283,2026-02-15 14:50:00,2026-02-15 15:08:00,"(SEAUX2,)",NaN,NaN,NaN,NaN,left_only,NaN,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Faltan Intervalos
4747,EE1026005461,2026-02-10 11:22:03,2026-02-10 21:14:17,"(SEAUX2, ZIG-ZAG)",NaN,NaN,NaN,NaN,left_only,NaN,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Faltan Intervalos
4748,EE1026006155,2026-02-15 02:06:30,2026-02-15 02:09:00,"(SEAUX2, ZIG-ZAG)",NaN,NaN,NaN,NaN,left_only,NaN,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Faltan Intervalos


,idinta,FECHA_INICIO,FECHA_FIN,SEDS_AFECTADAS,referencia,DURACION_MIN,Estado_x,Detalle Error,_merge,DIA,...,Estado_y,Fecha inicio-hora,Fecha Fin,Tipo Evento,Causa,Ticket,Cl. Afec.,G.res,Revision,Revision_vs_idinta
0,NaN,NaT,NaT,NaN,B020326J-09,NaN,NaN,NaN,NaN,2.0,...,CERRADO,2026-03-02 05:25:00,2026-03-02 10:58:00,AVERIA,INTERRUPCIÓN IMPREVISTA PUNTUAL EN J-09\nAPERT...,66,DF202681718012,P,REVISAR PARCIALMENTE,Faltan Intervalos
1,NaN,NaT,NaT,NaN,B02362A,NaN,NaN,NaN,NaN,5.0,...,CERRADO,2026-03-05 12:43:00,2026-03-05 15:16:00,DESCONEXION MANUAL O MANIOBRAS,CORTE POR EMERGENCIA EN V-08\nAPERTURA DE 0236...,300,DF202681721733,P,REVISAR PARCIALMENTE,Faltan Intervalos
2,NaN,NaT,NaT,NaN,B030326CV-03,NaN,NaN,NaN,NaN,3.0,...,CERRADO,2026-03-03 16:16:00,2026-03-03 18:07:00,DESCONEXION MANUAL O MANIOBRAS,CORTE POR EMERGENCIA EN CV-03\nAPERTURA DE 026...,2,DF202681719838,P,REVISAR PARCIALMENTE,Faltan Intervalos
3,NaN,NaT,NaT,NaN,B077663,NaN,NaN,NaN,NaN,1.0,...,CERRADO,2026-03-01 00:02:00,2026-03-01 00:04:00,DESCONEXION MANUAL O MANIOBRAS,APERTURA DE RP-01\nRETIRO DE G. ELECTROGENO E ...,494,DF202681717359,P,REVISAR PARCIALMENTE,Faltan Intervalos
4,NaN,2026-03-01 01:28:51,2026-03-01 02:57:07,"(02874A, 04733A, 04893A, 04894A, 04895A, 04896...",B077667,88.266667,CORRECTO,,right_only,1.0,...,CERRADO,2026-03-01 01:28:51,2026-03-01 17:12:00,AVERIA,INTERRUPCIÓN IMPREVISTA PARCIAL EN N-06\nAPERT...,1491,DF202681717364,T,VALIDAR,Faltan Intervalos
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4745,EE1026005021,2026-02-07 08:04:00,2026-02-07 11:33:14,"(PM2127,)",NaN,NaN,NaN,NaN,left_only,NaN,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Faltan Intervalos
4746,EE1026006283,2026-02-15 14:50:00,2026-02-15 15:08:00,"(SEAUX2,)",NaN,NaN,NaN,NaN,left_only,NaN,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Faltan Intervalos
4747,EE1026005461,2026-02-10 11:22:03,2026-02-10 21:14:17,"(SEAUX2, ZIG-ZAG)",NaN,NaN,NaN,NaN,left_only,NaN,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Faltan Intervalos
4748,EE1026006155,2026-02-15 02:06:30,2026-02-15 02:09:00,"(SEAUX2, ZIG-ZAG)",NaN,NaN,NaN,NaN,left_only,NaN,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Faltan Intervalos


In [14]:
referencias_validar = df_comparacion_Total[df_comparacion_Total["Revision"] == "VALIDAR"]
display(referencias_validar)
referencias_validar.to_excel(
    r"C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\GEMAS-RETIN\OUTPUT-TOTAL\Grupo-Correcto-Total.xlsx",
    index=False,
    engine="openpyxl"  # Asegura que se guarde correctamente en formato .xlsx
)

,idinta,FECHA_INICIO,FECHA_FIN,SEDS_AFECTADAS,referencia,DURACION_MIN,Estado_x,Detalle Error,_merge,DIA,...,Estado_y,Fecha inicio-hora,Fecha Fin,Tipo Evento,Causa,Ticket,Cl. Afec.,G.res,Revision,Revision_vs_idinta
4,NaN,2026-03-01 01:28:51,2026-03-01 02:57:07,"(02874A, 04733A, 04893A, 04894A, 04895A, 04896...",B077667,88.266667,CORRECTO,,right_only,1.0,...,CERRADO,2026-03-01 01:28:51,2026-03-01 17:12:00,AVERIA,INTERRUPCIÓN IMPREVISTA PARCIAL EN N-06\nAPERT...,1491,DF202681717364,T,VALIDAR,Faltan Intervalos
5,NaN,2026-03-01 02:57:07,2026-03-01 03:21:35,"(02874A, 04733A, 04893A, 04894A, 04895A, 04896...",B077667,24.466667,CORRECTO,,right_only,1.0,...,CERRADO,2026-03-01 01:28:51,2026-03-01 17:12:00,AVERIA,INTERRUPCIÓN IMPREVISTA PARCIAL EN N-06\nAPERT...,1491,DF202681717364,T,VALIDAR,Faltan Intervalos
6,EE1026008705,2026-03-01 03:21:35,2026-03-01 05:53:51,"(02874A, 04733A, 04893A, 04894A, 04895A, 04896...",B077667,152.266667,CORRECTO,,both,1.0,...,CERRADO,2026-03-01 01:28:51,2026-03-01 17:12:00,AVERIA,INTERRUPCIÓN IMPREVISTA PARCIAL EN N-06\nAPERT...,1491,DF202681717364,T,VALIDAR,Faltan Intervalos
7,EE1026008705,2026-03-01 05:53:51,2026-03-01 17:12:00,"(16297A,)",B077667,678.150000,CORRECTO,,both,1.0,...,CERRADO,2026-03-01 01:28:51,2026-03-01 17:12:00,AVERIA,INTERRUPCIÓN IMPREVISTA PARCIAL EN N-06\nAPERT...,1491,DF202681717364,T,VALIDAR,Faltan Intervalos
15,EE1026008799,2026-03-01 08:51:17,2026-03-01 16:30:17,"(51470U,)",B077698,459.000000,CORRECTO,,both,NaN,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,VALIDAR,Faltan Intervalos
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1031,EE1026013601,2026-03-29 07:00:36,2026-03-29 12:52:57,"(12581A, 12721A, 12967A, 12986A, 14339A, 14687...",B079370,352.350000,CORRECTO,,both,NaN,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,VALIDAR,Faltan Intervalos
1032,EE1026013623,2026-03-29 09:44:42,2026-03-29 14:51:24,"(PM0433,)",B079373,306.700000,CORRECTO,,both,NaN,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,VALIDAR,Faltan Intervalos
1033,EE1026013625,2026-03-29 10:01:00,2026-03-29 11:33:20,"(PM2045,)",B079374,92.333333,CORRECTO,,both,NaN,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,VALIDAR,Faltan Intervalos
1035,EE1026013633,2026-03-29 07:02:00,2026-03-29 20:00:00,"(04029A,)",B079376,778.000000,CORRECTO,,both,29.0,...,CERRADO,2026-03-29 07:02:00,2026-03-29 20:00:00,AVERIA,INTERRUPCIÓN IMPREVISTA PUNTUAL EN IE-08\nAPER...,DF202681745329,P,1063.27,VALIDAR,Faltan Intervalos


In [16]:
import pandas as pd

# Extraer valores únicos de la columna 'referencia'
valores_unicos = referencias_validar['REF CC'].unique()

# Convertir el array en un DataFrame
df_valores_unicos = pd.DataFrame(valores_unicos, columns=['REF CC'])

# Mostrar los valores únicos
display(df_valores_unicos)

# Exportar a Excel
df_valores_unicos.to_excel(
    r"C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\GEMAS-RETIN\OUTPUT-TOTAL\Valores_unicos.xlsx",
    index=False,
    engine="openpyxl"
)


,REF CC
0,B077667
1,NaN
2,B077708
3,B077710
4,B077724
...,...
205,B079352
206,B079363
207,B079365
208,B079376
